# 01 — Exploración del dataset combinado

Inventario y vista previa del dataset que usamos para entrenar la parte clásica:
**Klasson + Freiburg combinados en `data/external/combined/`** (7&nbsp;587 imágenes, 68 clases originales mapeadas a 8 categorías).

Este notebook responde a tres preguntas:
1. ¿Cuántas imágenes hay y cómo se distribuyen?
2. ¿Cómo se ven las imágenes de cada categoría?
3. ¿Hay clases mal mapeadas o anomalías visibles?

**Pre-requisitos:** ejecutar antes:

```bash
python scripts/download_klasson.py
python scripts/flatten_klasson.py
python scripts/download_freiburg_v2.py
python scripts/combine_datasets.py
```

> Cuando capturemos el dataset Monster, este notebook se complementará con un análisis específico de `data/raw/`.

In [ ]:
import sys
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from src.utils.io_utils import load_image, list_images
from src.classification.class_mapping import (
    map_klasson_class,
    PROJECT_CATEGORIES,
    KLASSON_TO_CATEGORY_FULL,
)

DATA_ROOT = Path('../data/external/combined')
plt.rcParams['figure.dpi'] = 90

## 1. Comprobación e inventario por clase original

Listamos todas las clases originales (Apple, BEANS, JUICE…) y cuántas imágenes tiene cada una.

In [ ]:
if not DATA_ROOT.is_dir():
    print(f'❌  No existe {DATA_ROOT}')
    print('Ejecuta: python scripts/combine_datasets.py')
else:
    inventory = {}
    for class_dir in sorted(DATA_ROOT.iterdir()):
        if class_dir.is_dir():
            inventory[class_dir.name] = list_images(class_dir)

    total = sum(len(v) for v in inventory.values())
    print(f'✓  Encontradas {len(inventory)} clases originales · {total} imágenes')
    print()
    print('Top 15 clases por número de imágenes:')
    for name, images in sorted(inventory.items(), key=lambda x: -len(x[1]))[:15]:
        cat = map_klasson_class(name)
        print(f'  {name:25s}  {len(images):4d}   →  {cat}')

## 2. Distribución por categoría del proyecto

Las 68 clases originales se mapean a las 8 categorías del proyecto. Esta es la distribución que ve el clasificador SVM.

In [ ]:
category_counts = Counter()
for class_name, images in inventory.items():
    cat = map_klasson_class(class_name)
    category_counts[cat] += len(images)

# Tabla
print(f'{"Categoría":<10}  {"Imágenes":>9}   Reparto')
print('─' * 45)
total = sum(category_counts.values())
for cat in PROJECT_CATEGORIES:
    n = category_counts.get(cat, 0)
    pct = 100 * n / total if total else 0
    bar = '█' * int(pct / 2)
    print(f'  {cat:<10s}  {n:5d}   {pct:5.1f}%  {bar}')

# Gráfica
fig, ax = plt.subplots(figsize=(9, 4))
cats = PROJECT_CATEGORIES
counts = [category_counts.get(c, 0) for c in cats]
colors = ['#FF6B00'] * len(cats)
ax.bar(cats, counts, color=colors, edgecolor='black')
ax.set_ylabel('Número de imágenes')
ax.set_title('Distribución por categoría del proyecto')
for i, v in enumerate(counts):
    ax.text(i, v + 30, str(v), ha='center', fontsize=9)
plt.tight_layout(); plt.show()

## 3. Detección de clases mal mapeadas

Si el script `combine_datasets.py` encontró clases que no están en `class_mapping.py`, irían a 'otros'. Comprobamos que no haya ninguna sorpresa.

In [ ]:
unmapped = [c for c in inventory if c not in KLASSON_TO_CATEGORY_FULL]
if not unmapped:
    print('✓  Todas las clases originales están mapeadas a una categoría del proyecto.')
else:
    print(f'⚠  {len(unmapped)} clases NO mapeadas (van a "otros"):')
    for c in unmapped:
        print(f'   - {c}  ({len(inventory[c])} imágenes)')
    print()
    print('Si alguna debería ir a otra categoría, edita src/classification/class_mapping.py')

## 4. Galería visual: una muestra de cada categoría del proyecto

In [ ]:
import random

n_per_cat = 4
fig, axes = plt.subplots(len(PROJECT_CATEGORIES), n_per_cat,
                         figsize=(3 * n_per_cat, 2.6 * len(PROJECT_CATEGORIES)))

for row, cat in enumerate(PROJECT_CATEGORIES):
    # Recoger imágenes de todas las clases originales que mapean a esta categoría
    imgs_for_cat = []
    for class_name, images in inventory.items():
        if map_klasson_class(class_name) == cat:
            imgs_for_cat.extend(images)

    sample = random.sample(imgs_for_cat, min(n_per_cat, len(imgs_for_cat))) if imgs_for_cat else []

    for col in range(n_per_cat):
        ax = axes[row, col]
        if col < len(sample):
            try:
                img = load_image(sample[col])
                ax.imshow(img)
            except Exception as e:
                ax.text(0.5, 0.5, f'Error\n{e}', ha='center', va='center', fontsize=7)
        ax.axis('off')
    axes[row, 0].set_ylabel(cat, fontsize=11, rotation=0, ha='right', va='center', fontweight='bold')

plt.suptitle('Una muestra aleatoria por categoría del proyecto', y=1.0, fontsize=12)
plt.tight_layout(); plt.show()

## 5. Distribución de tamaños de imagen

Útil para detectar imágenes anómalas (resoluciones raras) o decidir el tamaño de redimensionado para HOG y LBP.

In [ ]:
# Muestra rápida: 200 imágenes aleatorias para no leer las 7500 de golpe
all_images = []
for images in inventory.values():
    all_images.extend(images)

sample_size = min(200, len(all_images))
sample_paths = random.sample(all_images, sample_size)

sizes = []
for path in sample_paths:
    try:
        img = load_image(path)
        h, w = img.shape[:2]
        sizes.append((w, h))
    except Exception:
        pass

if sizes:
    widths, heights = zip(*sizes)
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].hist(widths, bins=20, color='#FF6B00', edgecolor='black', alpha=0.85)
    axes[0].set_title(f'Anchos (px) · muestra de {len(sizes)} imágenes')
    axes[0].set_xlabel('Píxeles')
    axes[1].hist(heights, bins=20, color='#0A0A0A', edgecolor='black', alpha=0.85)
    axes[1].set_title(f'Altos (px) · muestra de {len(sizes)} imágenes')
    axes[1].set_xlabel('Píxeles')
    plt.tight_layout(); plt.show()
    print(f'Ancho:  min={min(widths)}, max={max(widths)}, media={np.mean(widths):.0f}')
    print(f'Alto:   min={min(heights)}, max={max(heights)}, media={np.mean(heights):.0f}')

## 6. Conclusiones

Lo que esperamos confirmar al ejecutar:

- **No hay clases sin mapear** (sección 3). Si las hay, hay que decidir adónde van.
- **Las 8 categorías están todas representadas**, aunque desbalanceadas (sección 2). El SVM compensa el desbalance con `class_weight='balanced'`.
- **Las muestras visuales son razonables** (sección 4): productos centrados, iluminación de tienda real.

**Próximo paso:** `02b_preprocesado_completo.ipynb` para ver cómo el preprocesado mejora estas imágenes.